In [1]:
import os
import sys
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.master("local[*]").appName("limpieza").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 3.5.9


In [2]:
esquema = StructType([
    StructField("id_venta", IntegerType()),
    StructField("cliente", StringType()),
    StructField("producto", StringType()),
    StructField("color", StringType()),
    StructField("vendedor", StringType()),
    StructField("monto", IntegerType()),
    StructField("fecha", StringType()),
])

ruta = Path("../data/ventas_pentagrama_sim.csv").resolve().as_uri()
df = spark.read.csv(ruta, header=True, schema=esquema)

print(df.count())
df.printSchema()

205
root
 |-- id_venta: integer (nullable = true)
 |-- cliente: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- color: string (nullable = true)
 |-- vendedor: string (nullable = true)
 |-- monto: integer (nullable = true)
 |-- fecha: string (nullable = true)



In [3]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+--------+-------+--------+-----+--------+-----+-----+
|id_venta|cliente|producto|color|vendedor|monto|fecha|
+--------+-------+--------+-----+--------+-----+-----+
|       0|      8|       0|    0|       0|    0|    0|
+--------+-------+--------+-----+--------+-----+-----+



In [4]:
df.groupBy(df.columns).count().filter(F.col("count") > 1).show()

+--------+--------------------+-------------------+-----+------------+------+----------+-----+
|id_venta|             cliente|           producto|color|    vendedor| monto|     fecha|count|
+--------+--------------------+-------------------+-----+------------+------+----------+-----+
|     132|     Patricia Ortega|Persiana enrollable|Negro|Luis Ramirez|204348|2026-07-05|    2|
|      83|Viviana Carolina ...|   Cortina blackout| Azul| Sofia Gomez|449838|2026-04-02|    2|
|      65| Lucía Ramírez Ávila|Cortina translucida| Azul|  Ana Torres|116231|2026-05-20|    2|
|      37|   Alfredo Hernández|Cortina translucida| Azul|Luis Ramirez|282825|2026-05-31|    2|
|      75| Ferney Yesid Santos|Cortina translucida| Gris|  Ana Torres|382873|2026-04-13|    2|
+--------+--------------------+-------------------+-----+------------+------+----------+-----+



In [5]:
def limpiar_ventas(df):
    return (
        df.dropna(subset=["cliente"])
        .dropDuplicates(["cliente", "producto", "monto", "fecha"])
        .withColumn("monto", F.col("monto").cast("int"))
        .withColumn("fecha", F.to_date("fecha"))
    )

limpio = limpiar_ventas(df)
print("Antes:", df.count(), "| Después:", limpio.count())
limpio.printSchema()
limpio.show(5)

Antes: 205 | Después: 192
root
 |-- id_venta: integer (nullable = true)
 |-- cliente: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- color: string (nullable = true)
 |-- vendedor: string (nullable = true)
 |-- monto: integer (nullable = true)
 |-- fecha: date (nullable = true)

+--------+--------------------+-------------------+-----+------------+------+----------+
|id_venta|             cliente|           producto|color|    vendedor| monto|     fecha|
+--------+--------------------+-------------------+-----+------------+------+----------+
|     136| Adolfo Luis Arteaga|Cortina translucida| Gris|  Ana Torres|468451|2026-09-15|
|     109|Adrián Jorge Cruz...|      Panel japones|Negro|Luis Ramirez|136080|2026-04-29|
|      91|Albeiro Luis Beja...|      Panel japones|Beige|  Ana Torres|164374|2026-08-11|
|      94|Alberto Gutiérrez...| Persiana veneciana| Azul|Luis Ramirez|378551|2026-09-08|
|     192|    Alberto Portilla|      Panel japones|Negro|Luis Ramirez|49111

In [6]:
sin_nulos = df.dropna(subset=["cliente"])
n_nulos = df.count() - sin_nulos.count()
n_dups = sin_nulos.count() - sin_nulos.dropDuplicates(["cliente", "producto", "monto", "fecha"]).count()
print("Filas sin cliente:", n_nulos, "| Duplicadas:", n_dups, "| Total eliminado:", n_nulos + n_dups)

Filas sin cliente: 8 | Duplicadas: 5 | Total eliminado: 13


In [7]:
import pandas as pd

pdf = pd.read_csv("../data/ventas_pentagrama_sim.csv")
pdf = pdf.dropna(subset=["cliente"]).drop_duplicates(subset=["cliente", "producto", "monto", "fecha"])
print("pandas:", len(pdf), int(pdf["monto"].sum()))

r = limpio.agg(F.count("*").alias("filas"), F.sum("monto").alias("total")).collect()[0]
print("Spark :", r["filas"], r["total"])

pandas: 192 60407192
Spark : 192 60407192


In [8]:
limpio_pd = limpio.toPandas()
print(limpio_pd.shape)
print(limpio_pd.dtypes)

(192, 7)
id_venta     int32
cliente     object
producto    object
color       object
vendedor    object
monto        int32
fecha       object
dtype: object


In [9]:
spark.stop()